In this file I will attempt to work on the GNR approach to production estimation

In [155]:
# Import packages 
using Pkg 
using GLM
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using Random
using Distributions
using Optim
using Plots
using ShiftedArrays  # for lag function

In [156]:
# [NOTE TO SELF]: save the value-added calculation for later and just to the raw dataset for now

# Step 0: preparing the dataset - with material share variable 
dataset = CSV.read("gnr_ready.csv", DataFrame)

display(first(dataset, 10))

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,v_m_share
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1997,15.2437,12.2967,13.6469,0.0199931,27.4075,-0.463193
2,1,1998,15.4399,12.6583,13.8412,0.0199931,20.8425,-0.417266
3,1,1999,15.3679,12.3587,13.9283,0.0199931,25.9536,-0.393546
4,2,1997,14.8474,12.3572,13.6883,-0.00911877,43.5369,-0.56622
5,2,1998,14.7601,11.5891,13.6968,-0.0191691,41.273,-0.523192
6,2,1999,14.7038,12.6857,13.8871,-0.0292194,42.8179,-0.621868
7,3,1998,13.9124,12.1451,12.0143,0.145003,17.4865,-0.241295
8,3,1999,13.66,9.89273,12.0052,0.184223,17.4949,-0.227456
9,4,1995,17.6699,14.5626,17.1751,0.0532712,71.6182,-0.42778


Step 1: re-cover and calibrate the polynomial of capital, labor, and materials 

- Using non-linear least square to do this: minimizing the square of ([material_share] - [fitted polynomial of k,l, and m])

In [157]:
function h_polynomial(parameters, capital, labor, material) # return fitted value for given linear parameters 
    alpha_0, alpha_1, alpha_2, alpha_3, alpha_4, alpha_5, alpha_6, alpha_7, alpha_8, alpha_9 = parameters
    sum =  alpha_0 .+ alpha_1.*capital .+ alpha_2.*labor .+ alpha_3*material .+ 
                    alpha_4*capital.^2 .+ alpha_5*labor.^2 + alpha_6*material.^2 .+ 
                    alpha_7*capital.*labor .+ alpha_8*capital.*material + alpha_9*labor.*material
    return sum
end

function loss_function(parameters, dataset)
    fitted_values = h_polynomial(parameters, dataset.v_capital, dataset.v_labor, dataset.v_material)
    residuals = dataset.v_m_share .- fitted_values
    # "might be useful to explicitly punish negative inside logs" <- come back to this 
    return sum(residuals.^2)
end

println("Test loss function output: ", loss_function(ones(10), dataset))

function NLLS(initial_parameters, dataset)
    result = optimize(linear_param -> loss_function(linear_param, dataset), initial_parameters, LBFGS())
    min_loss = Optim.minimum(result)   # minimum loss value
    println("Minimum loss value: ", min_loss)
    return Optim.minimizer(result)
end

estimated_linear = NLLS(ones(10),dataset)
println("Un-scaled linear parameters:" , estimated_linear)
# now we have our estimated linear parameters, just need to re-calibrae it by scaling by the residuals 

# recover the residuals by 
residuals = -(dataset.v_m_share .- h_polynomial(estimated_linear, dataset.v_capital, dataset.v_labor, dataset.v_material))

# print("Residual terms: ", residuals)

# recover the scalar by summing over exponents of the residuals 
rescaler = sum(exp.(residuals)) / length(residuals)

# now we re-scale the linear parameters
scaled_parameters = estimated_linear./rescaler

print("Scaled parameters: ", scaled_parameters)
# now we have the scaled parameters, so onto step 2! 

Test loss function output: 4.796308706728053e10
Minimum loss value: 1271.5402026251875
Un-scaled linear parameters:[-1.49616589027981, 0.12294973237828394, -0.0010276576685132315, -0.36122092846448456, -0.0033084830841831215, 1.1326985589449515e-6, 0.7066903265152755, 1.6990452927870204e-6, 0.06492666197170015, -0.004524771518859737]
Scaled parameters: [-1.3838344396016944, 0.11371872270999389, -0.0009505016008241116, -0.3341006263821247, -0.0030600836875622085, 1.047656069249172e-6, 0.6536323400490783, 1.5714817493681176e-6, 0.06005199788909314, -0.0041850537429081154]

Now we can obtain the production function by intergration over this intergral analytically to get the value up to an additive term, after that we can use GMM to recover this additive terms (representing productivity)

So there are two smaller steps: (1) analytical integration and (2) GMM to recover additive terms

In [158]:
# B1: analytical integration and calculate the RHS of the equation 
function integration(estimated_parameters, dataset)
    capital, labor, material = dataset.v_capital, dataset.v_labor, dataset.v_material
    alpha_0, alpha_1, alpha_2, alpha_3, alpha_4, alpha_5, alpha_6, alpha_7, alpha_8, alpha_9 = estimated_parameters
    linear_term = material.*(alpha_0 .+ alpha_1.*capital .+ alpha_2.*labor .+ alpha_4.*capital.^2 .+ alpha_5*labor.^2 .+ alpha_7*capital.*labor)

    # m term integrates to (1/2) * m^2
    m_term = alpha_3 .* material.^2 ./ 2

    # m^2 term integrates to (1/3) * m^3
    m2_term = alpha_6 .* material.^3 ./ 3

    # cross terms with m: km and lm integrate to km*m^2/2 and lm*m^2/2
    km_term = alpha_8 .* capital .* material.^2 ./ 2
    lm_term = alpha_9 .* labor .* material.^2 ./ 2
    
    integrated_term = linear_term .+ m_term .+ m2_term .+ km_term .+ lm_term
    return integrated_term 
end

integrated = integration(scaled_parameters, dataset)

LHS = dataset.v_production .- residuals .- integrated

dataset.LHS = LHS # now square terms
dataset.integrated = integrated
dataset.residuals = residuals

# display(first(dataset,10))

sort!(dataset, [:firm_id, :year])
transform!(groupby(dataset, :firm_id), :LHS => (x -> lag(x, 1)) => :lag_LHS) #generate capital lag variable
transform!(groupby(dataset, :firm_id), :v_capital => (x -> lag(x, 1)) => :lag_capital) # generate phi lag variable
transform!(groupby(dataset, :firm_id), :v_labor => (x -> lag(x, 1)) => :lag_labor) # generate material lag labor

dataset = dropmissing(dataset, [:lag_capital, :lag_LHS, :lag_labor])

display(first(dataset, 10))
# now we have most of the parameters, except for the additive terms which we will use GMM to recover 

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,v_m_share,LHS,integrated,residuals,lag_LHS,lag_capital,lag_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1998,15.4399,12.6583,13.8412,0.0199931,20.8425,-0.417266,15.4704,-0.00821311,-0.0222588,15.2418,13.6469,27.4075
2,1,1999,15.3679,12.3587,13.9283,0.0199931,25.9536,-0.393546,15.4247,-0.00825609,-0.0484768,15.4704,13.8412,20.8425
3,2,1998,14.7601,11.5891,13.6968,-0.0191691,41.273,-0.523192,14.7071,0.00843119,0.044558,14.7548,13.6883,43.5369
4,2,1999,14.7038,12.6857,13.8871,-0.0292194,42.8179,-0.621868,14.5464,0.0127717,0.14463,14.7071,13.6968,41.273
5,3,1999,13.66,9.89273,12.0052,0.184223,17.4949,-0.227456,13.9413,-0.0809384,-0.200335,14.1857,12.0143,17.4865
6,4,1996,17.6065,15.2652,17.2278,0.08283,78.801,-0.420059,17.6116,-0.03167,0.026503,17.6638,17.1751,71.6182
7,5,1999,16.0031,14.0184,15.313,0.10276,51.9997,-0.321418,16.1111,-0.0394754,-0.0685075,16.1726,15.1472,52.9994
8,6,1999,11.0108,8.07091,8.89698,0.0272698,-4.84692,-1.80719,9.8724,-0.0165411,1.15498,9.91107,8.54161,-5.17222
9,7,1997,12.5013,8.65869,9.51618,0.0,16.5183,-0.513653,12.6298,0.0,-0.128508,12.1473,9.18124,14.1084


In [ ]:
# B2: GMM to recover additive terms: defining the polynomial fit for RHS, fit it to produce loss function, GMM with the loss function
function fitted_RHS(parameter_guess,     
    capital, lag_capital,
    labor, lag_labor,
    LHS, lag_LHS)

    alpha_1, alpha_2, alpha_3, alpha_4, alpha_5, beta_0, beta_1 = parameter_guess # 5 linear poly terms for (k,l) and 2 for markov process

    C_kl_term = alpha_1 .* capital .+ alpha_2 .* labor .+ alpha_3 .* capital.^2 .+ alpha_4 .* labor.^2 .+ alpha_5 .* capital .* labor
    markov_term = beta_0 .+ beta_1 .* (lag_LHS .- 
                        (alpha_1 .* lag_capital .+ alpha_2 .* lag_labor .+ 
                        alpha_3 .* lag_capital.^2 .+ alpha_4 .* lag_labor.^2 .+ alpha_5 .* lag_capital .* lag_labor))

    RHS = C_kl_term .+ markov_term 
    return RHS
end # return the RHS term 

function objective_function(parameter_guess,
    capital, lag_capital,
    labor, lag_labor,
    LHS, lag_LHS
) # return loss value for GMM to minimize
    RHS = fitted_RHS(parameter_guess,
        capital, lag_capital,
        labor, lag_labor,
        LHS, lag_LHS)
    epsilon = LHS .- RHS # calculate the residuals and thus GMM loss value 

    # now the instrument stuff
    instrument = Matrix(hcat(capital, lag_capital, labor, lag_labor, lag_LHS))  # Nx5 matrix 
    g = (transpose(epsilon) * instrument) ./ length(instrument)
    loss = dot(g, g)  # equivalent to g'T * g

    return loss
end

function GMM_main(
    initial_guess, 
    dataset
)
    result = optimize(parameter_guess -> objective_function(
            parameter_guess,
            dataset.v_capital, dataset.lag_capital,
            dataset.v_labor, dataset.lag_labor,
            dataset.LHS, dataset.lag_LHS
            ),
        initial_guess,
        NelderMead()
    )
    println("Initial loss value:", objective_function(initial_guess,
            dataset.v_capital, dataset.lag_capital,
            dataset.v_labor, dataset.lag_labor,
            dataset.LHS, dataset.lag_LHS))
    min_loss = Optim.minimum(result)   # minimum loss value
    println("Minimum loss value: ", min_loss)

    parameter_estimated = Optim.minimizer(result)
    println("Estimated parameter:", parameter_estimated)
    return parameter_estimated
end 


step2_params = GMM_main(ones(7), dataset)

Initial loss value:1.725037990687177e6
Minimum loss value: 0.014444096230670308
Estimated parameter:[1.6777739062898922, 1.8016241097080496, 0.20599525773946312, 0.011131203255840367, -0.1826604771729758, 1.7276136318926236, 1.0589845514974057]


7-element Vector{Float64}:
  1.6777739062898922
  1.8016241097080496
  0.20599525773946312
  0.011131203255840367
 -0.1826604771729758
  1.7276136318926236
  1.0589845514974057

In [160]:
# now that all parameters have been estimated, we put things back together and see if the predicted production matches the actual production
pred_RHS = fitted_RHS(
    step2_params,
    dataset.v_capital, dataset.lag_capital,
    dataset.v_labor, dataset.lag_labor,
    dataset.LHS, dataset.lag_LHS
)
# now append them onto our dataset 
dataset.pred_production = pred_RHS + dataset.integrated + dataset.residuals


display(first(dataset, 20))

CSV.write("exported_GNR.csv", dataset)

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,v_m_share,LHS,integrated,residuals,lag_LHS,lag_capital,lag_labor,pred_production
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1998,15.4399,12.6583,13.8412,0.0199931,20.8425,-0.417266,15.4704,-0.00821311,-0.0222588,15.2418,13.6469,27.4075,16.5463
2,1,1999,15.3679,12.3587,13.9283,0.0199931,25.9536,-0.393546,15.4247,-0.00825609,-0.0484768,15.4704,13.8412,20.8425,14.1444
3,2,1998,14.7601,11.5891,13.6968,-0.0191691,41.273,-0.523192,14.7071,0.00843119,0.044558,14.7548,13.6883,43.5369,13.7669
4,2,1999,14.7038,12.6857,13.8871,-0.0292194,42.8179,-0.621868,14.5464,0.0127717,0.14463,14.7071,13.6968,41.273,14.6874
5,3,1999,13.66,9.89273,12.0052,0.184223,17.4949,-0.227456,13.9413,-0.0809384,-0.200335,14.1857,12.0143,17.4865,13.6991
6,4,1996,17.6065,15.2652,17.2278,0.08283,78.801,-0.420059,17.6116,-0.03167,0.026503,17.6638,17.1751,71.6182,19.555
7,5,1999,16.0031,14.0184,15.313,0.10276,51.9997,-0.321418,16.1111,-0.0394754,-0.0685075,16.1726,15.1472,52.9994,15.1724
8,6,1999,11.0108,8.07091,8.89698,0.0272698,-4.84692,-1.80719,9.8724,-0.0165411,1.15498,9.91107,8.54161,-5.17222,13.9157
9,7,1997,12.5013,8.65869,9.51618,0.0,16.5183,-0.513653,12.6298,0.0,-0.128508,12.1473,9.18124,14.1084,14.2589


"exported_GNR.csv"